In [1]:
import os
os.chdir("../../")
import pandas as pd
from typing import List
import json

from alti_analysis.utils import create_directory_structure

In [2]:
def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output

def read_jsonl(fname):
    output = []
    with open(fname, 'r', encoding='utf-8') as file:
        for line in file:
            output.append(json.loads(line))
    return output

def read_json(fname):
    with open(fname, 'r', encoding='utf-8') as file:
            output = json.load(file)
    return output


def get_instructions(lp, context_type, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    
    instructions = read_file(f"instructions/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/instructions.txt")    
    return instructions


def get_generations(lp, context_type,model_name, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    generations = read_file(f"generations/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/vllm/{model_name}/generation.txt")    
    return generations


def get_pcxmi_log_probs(lp, context_type,model_name, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    #read hypothesis logprobs!
    probs_hyp = read_jsonl(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/log_probs_hyp.jsonl")
    probs_ref = read_jsonl(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/log_probs_ref.jsonl")
    assert len(probs_hyp)==len(probs_ref),"Log probs lengths for hyp and ref not equal"
    return {"hyp":probs_hyp,"ref":probs_ref}


def get_pcxmi_sentence_mean_log_probs(lp, context_type,model_name, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    #read  logprobs!
    probs_hyp = read_file(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/mean_log_probs_hyp.txt")
    probs_ref = read_file(f"pcxmi/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/{model_name}/mean_log_probs_ref.txt")
    assert  len(probs_hyp)==len(probs_ref), "Log probs lengths for hyp and ref not equal"
    return {"hyp":probs_hyp,"ref":probs_ref}


def get_eval_comet_scores(lp, context_type,model_name, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    #read comet_kiwi_scores for each turn
    eval_data = read_json(f"evaluations/{context_type}/mt/wmt24_chat_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/vllm/{model_name}/evaluation.json")
    return eval_data["comet_kiwi_segments"]



In [11]:
langs = ["de-en","fr-en","pt-en","ko-en","nl-en","en-de","en-fr","en-pt","en-ko","en-nl"]
split="test"

#define model name

##### TowerInstruct-7B-v0.2  with full context   ##########
# model_name = "TowerInstruct-7B-v0.2"
# context_type = "full_context"
###########################################################


##### TowerInstruct-7B-v0.2  with no context   ##########
# model_name = "TowerInstruct-7B-v0.2" 
# context_type = "no_context"
###########################################################


###### TowerChat  with full context   ##########
# model_name = "TowerInstruct-7B-w-chat-empty-sys" 
# context_type = "full_context_empty_sys"
###########################################################


###### TowerChat  with no context   ##########
# model_name = "TowerInstruct-7B-w-chat-empty-sys" 
# context_type = "no_context_empty_sys"
###########################################################




savepath = f"./alti_prep_data/v2/{context_type}/{model_name}/"

pcxmi_log_probs=True
pcxmi_sentence_log_probs=True
eval_comet=True


data_dict = {}
for lp in langs:
    data_dict[lp]= {}
    tmp_dict = {}
    #read instructions
    tmp_dict["instruction"] = get_instructions(lp, context_type, split)
    tmp_dict["generation"] = get_generations(lp, context_type,model_name, split)
    if pcxmi_log_probs:
        log_probs = get_pcxmi_log_probs(lp, context_type,model_name, split)
        tmp_dict["log_probs_hyp"] = log_probs["hyp"]
        tmp_dict["log_probs_ref"] = log_probs["hyp"]
    if pcxmi_sentence_log_probs:
        log_probs = get_pcxmi_sentence_mean_log_probs(lp, context_type,model_name, split)
        tmp_dict["mean_probs_hyp"] = log_probs["hyp"]
        tmp_dict["mean_probs_ref"] = log_probs["hyp"]
    if eval_comet:
        tmp_dict["comet_score"] = get_eval_comet_scores(lp, context_type,model_name, split)
    df = pd.DataFrame(tmp_dict)
    if context_type!="no_context":
        #if no context then we should exclude all rows not containing context! 
        df = df.drop(df[~df['instruction'].apply(lambda x : '\\nContext:' in x)].index)
        df['id'] = df.index
    my_list = df.to_dict(orient='records')
    data_dict[lp] = my_list



In [12]:
for lp in data_dict.keys():
    fname = os.path.join(savepath, f"{split}_{lp}.json")
    create_directory_structure(fname)
    data = data_dict[lp]
    with open(fname, 'w') as f:
        json.dump(data, f)